In [ ]:
# =========================================================
# Super-Resolution on Kaggle (P100-compatible)
# Role: ML Engineer + Computer Vision Specialist
# Objective: LR -> HR with Bayesian optimization + ablations
# =========================================================

# Notebook flow:
# 1) Environment and dependency setup
# 2) Dataset with augmentations/degradations
# 3) Efficient SR model (FastEDSR)
# 4) Bayesian hyperparameter optimization (Optuna)
# 5) Ablation studies
# 6) Final training: 200 epochs, patience 20, dynamic LR


In [ ]:
# Kaggle-safe dependency install (skip if already available)
%pip -q install optuna tqdm albumentations --upgrade

In [ ]:
# Optional: if you still see CUDA compatibility issues on older Kaggle images, uncomment and run, then restart kernel.
# %pip -q install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import os
import random
import gc
import json
import math
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

import albumentations as A
from tqdm.auto import tqdm
import optuna

# Reproducibility
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print("GPU:", name, "| CUDA capability:", cap)
    if cap[0] < 6:
        raise RuntimeError("GPU capability is too low for practical SR training.")

In [ ]:
# --------------------------
# Config
# --------------------------
@dataclass
class CFG:
    image_dir: str = "/kaggle/input/joe1995-div2k-dataset/DIV2K_train_HR/DIV2K_train_HR"
    save_dir: str = "/kaggle/working/sr_outputs"
    scale: int = 4
    hr_size: int = 128  # HR patch size
    batch_size: int = 16
    # Use single-process loading in notebooks to avoid multiprocessing shutdown errors.
    num_workers: int = 0
    train_ratio: float = 0.9

    # Search settings
    n_trials: int = 12
    search_epochs: int = 12

    # Final training settings
    final_epochs: int = 200
    patience: int = 20

cfg = CFG()
os.makedirs(cfg.save_dir, exist_ok=True)
print(cfg)

In [ ]:
# --------------------------
# SR Dataset with augmentations and realistic degradations
# --------------------------
class SRDataset(Dataset):
    def __init__(self, image_dir, hr_size=128, scale=4, use_augmentation=True):
        valid_ext = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
        self.image_paths = [
            os.path.join(image_dir, x)
            for x in os.listdir(image_dir)
            if x.lower().endswith(valid_ext)
        ]
        self.image_paths.sort()
        self.hr_size = hr_size
        self.lr_size = hr_size // scale
        self.scale = scale
        self.use_augmentation = use_augmentation

        self.base_hr = A.Compose([
            A.SmallestMaxSize(max_size=hr_size + 32, p=1.0),
            A.RandomCrop(height=hr_size, width=hr_size, p=1.0),
        ])

        self.aug_hr = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.1),
            A.RandomRotate90(p=0.25),
            A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.08, p=0.4),
        ])

    def __len__(self):
        return len(self.image_paths)

    def _degrade_to_lr(self, hr_np):
        # Bicubic downscale + mild blur/noise to mimic practical degradations
        lr = A.Resize(self.lr_size, self.lr_size, interpolation=cv2.INTER_CUBIC, p=1.0)(image=hr_np)["image"]
        if self.use_augmentation:
            if np.random.rand() < 0.3:
                sigma = np.random.uniform(0.2, 1.2)
                lr = A.GaussianBlur(blur_limit=(3, 5), sigma_limit=(sigma, sigma), p=1.0)(image=lr)["image"]
            if np.random.rand() < 0.3:
                noise_std = np.random.uniform(2.0, 8.0)
                noise = np.random.normal(0, noise_std, lr.shape).astype(np.float32)
                lr = np.clip(lr.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        return lr

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img_np = np.array(img)

        hr_np = self.base_hr(image=img_np)["image"]
        if self.use_augmentation:
            hr_np = self.aug_hr(image=hr_np)["image"]

        lr_np = self._degrade_to_lr(hr_np)

        hr = torch.from_numpy(hr_np).permute(2, 0, 1).float() / 255.0
        lr = torch.from_numpy(lr_np).permute(2, 0, 1).float() / 255.0
        return lr, hr

Device: cuda
CUDA: True
GPU: Tesla P100-PCIE-16GB


In [ ]:
# --------------------------
# Fast EDSR model (LR -> HR)
# --------------------------
class LiteResBlock(nn.Module):
    def __init__(self, n_feats, res_scale=0.1):
        super().__init__()
        self.dw = nn.Conv2d(n_feats, n_feats, 3, 1, 1, groups=n_feats)
        self.pw = nn.Conv2d(n_feats, n_feats, 1, 1, 0)
        self.act = nn.ReLU(inplace=True)
        self.res_scale = res_scale

    def forward(self, x):
        res = self.act(self.dw(x))
        res = self.pw(res)
        return x + res * self.res_scale


class Upsampler(nn.Sequential):
    def __init__(self, scale, n_feats):
        layers = []
        if scale in [2, 4, 8]:
            for _ in range(int(np.log2(scale))):
                layers += [nn.Conv2d(n_feats, 4 * n_feats, 3, 1, 1), nn.PixelShuffle(2)]
        elif scale == 3:
            layers += [nn.Conv2d(n_feats, 9 * n_feats, 3, 1, 1), nn.PixelShuffle(3)]
        else:
            raise ValueError(f"Unsupported scale {scale}")
        super().__init__(*layers)


class FastEDSR(nn.Module):
    def __init__(self, scale=4, n_resblocks=16, n_feats=64, n_colors=3, res_scale=0.1):
        super().__init__()
        self.head = nn.Conv2d(n_colors, n_feats, 3, 1, 1)
        self.body = nn.Sequential(*[LiteResBlock(n_feats, res_scale) for _ in range(n_resblocks)])
        self.body_tail = nn.Conv2d(n_feats, n_feats, 3, 1, 1)
        self.tail = nn.Sequential(
            Upsampler(scale, n_feats),
            nn.Conv2d(n_feats, n_colors, 3, 1, 1),
        )

    def forward(self, x):
        x = self.head(x)
        res = self.body(x)
        res = self.body_tail(res)
        x = x + res
        x = self.tail(x)
        return x

In [ ]:
# --------------------------
# Training / Eval Utilities
# --------------------------
def psnr_torch(sr, hr, eps=1e-12):
    mse = F.mse_loss(sr, hr, reduction="none").flatten(1).mean(dim=1)
    return (10.0 * torch.log10(1.0 / torch.clamp(mse, min=eps))).mean().item()


class EarlyStopping:
    def __init__(self, patience=20, mode="max", min_delta=1e-4):
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.best = None
        self.counter = 0

    def step(self, metric):
        if self.best is None:
            self.best = metric
            return False

        improved = (metric > self.best + self.min_delta) if self.mode == "max" else (metric < self.best - self.min_delta)
        if improved:
            self.best = metric
            self.counter = 0
            return False

        self.counter += 1
        return self.counter >= self.patience


def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None, use_amp=True):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader, desc="train", leave=False)
    for lr, hr in pbar:
        lr = lr.to(device, non_blocking=True)
        hr = hr.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        if use_amp and device.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                sr = model(lr)
                loss = criterion(sr, hr)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            sr = model(lr)
            loss = criterion(sr, hr)
            loss.backward()
            optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / max(len(loader), 1)


@torch.inference_mode()
def evaluate(model, loader, criterion, device, use_amp=True):
    model.eval()
    total_loss = 0.0
    total_psnr = 0.0
    pbar = tqdm(loader, desc="valid", leave=False)
    for lr, hr in pbar:
        lr = lr.to(device, non_blocking=True)
        hr = hr.to(device, non_blocking=True)

        if use_amp and device.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                sr = model(lr)
                loss = criterion(sr, hr)
        else:
            sr = model(lr)
            loss = criterion(sr, hr)

        sr = sr.float().clamp(0, 1)
        total_loss += loss.item()
        total_psnr += psnr_torch(sr, hr.float())

    avg_loss = total_loss / max(len(loader), 1)
    avg_psnr = total_psnr / max(len(loader), 1)
    return avg_loss, avg_psnr


def save_model(path, model, meta):
    torch.save({
        "model_state_dict": model.state_dict(),
        "meta": meta,
    }, path)

Dataset size: 800
Epoch 1/3 - Loss: 0.2376
Epoch 2/3 - Loss: 0.1259
Epoch 3/3 - Loss: 0.1044
Saved checkpoint to /kaggle/working/edsr_x4.pth


NameError: name 'EDSR' is not defined

In [ ]:
# --------------------------
# Data preparation + sample visualization
# --------------------------
all_ds = SRDataset(
    image_dir=cfg.image_dir,
    hr_size=cfg.hr_size,
    scale=cfg.scale,
    use_augmentation=True,
)

n_total = len(all_ds)
n_train = int(cfg.train_ratio * n_total)
n_val = n_total - n_train
train_ds, val_ds = random_split(all_ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))

# Keep DataLoader single-process in notebooks to avoid multiprocessing cleanup errors.
train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=False,
    drop_last=True,
    generator=torch.Generator().manual_seed(42),
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=False,
    drop_last=False,
)

print(f"Total: {n_total}, Train: {len(train_ds)}, Val: {len(val_ds)}")

lr_ex, hr_ex = all_ds[0]
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(lr_ex.permute(1, 2, 0).numpy()); ax[0].set_title("LR")
ax[1].imshow(hr_ex.permute(1, 2, 0).numpy()); ax[1].set_title("HR")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------
# Bayesian Optimization (Optuna)
# --------------------------
def build_model_from_trial(trial):
    n_resblocks = trial.suggest_int("n_resblocks", 8, 24, step=4)
    n_feats = trial.suggest_categorical("n_feats", [32, 48, 64, 80])
    res_scale = trial.suggest_float("res_scale", 0.05, 0.2)
    return FastEDSR(scale=cfg.scale, n_resblocks=n_resblocks, n_feats=n_feats, res_scale=res_scale).to(device)


def objective(trial):
    lr_rate = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-8, 1e-4, log=True)
    loss_name = trial.suggest_categorical("loss", ["l1", "charbonnier"])

    model = build_model_from_trial(trial)
    criterion = nn.L1Loss() if loss_name == "l1" else (lambda x, y: torch.mean(torch.sqrt((x - y) ** 2 + 1e-6)))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_rate, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=4, min_lr=1e-6
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    early_stop = EarlyStopping(patience=6, mode="max")

    best_psnr = -1.0
    for epoch in range(cfg.search_epochs):
        train_loss = train_one_epoch(
            model=model, loader=train_loader, optimizer=optimizer, criterion=criterion,
            device=device, scaler=scaler, use_amp=True
        )
        val_loss, val_psnr = evaluate(model, val_loader, criterion, device, use_amp=True)
        scheduler.step(val_psnr)

        trial.report(val_psnr, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        best_psnr = max(best_psnr, val_psnr)
        if early_stop.step(val_psnr):
            break

    del model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return best_psnr


sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=3)
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=cfg.n_trials)

print("Best trial value (PSNR):", study.best_value)
print("Best params:", study.best_trial.params)

best_params_path = os.path.join(cfg.save_dir, "best_optuna_params.json")
with open(best_params_path, "w", encoding="utf-8") as f:
    json.dump(study.best_trial.params, f, indent=2)
print("Saved best params to", best_params_path)

In [ ]:
# --------------------------
# Ablation study
# --------------------------
def make_loaders(use_aug=True, batch_size=None):
    ds = SRDataset(cfg.image_dir, hr_size=cfg.hr_size, scale=cfg.scale, use_augmentation=use_aug)
    nt = len(ds)
    ntr = int(cfg.train_ratio * nt)
    nvl = nt - ntr
    tr, vl = random_split(ds, [ntr, nvl], generator=torch.Generator().manual_seed(42))
    bs = batch_size or cfg.batch_size

    tr_loader = DataLoader(
        tr,
        batch_size=bs,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=False,
        drop_last=True,
    )
    vl_loader = DataLoader(
        vl,
        batch_size=bs,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=False,
        drop_last=False,
    )
    return tr_loader, vl_loader


def run_ablation(name, use_aug, use_scheduler, epochs=10):
    p = study.best_trial.params
    model = FastEDSR(
        scale=cfg.scale,
        n_resblocks=int(p["n_resblocks"]),
        n_feats=int(p["n_feats"]),
        res_scale=float(p["res_scale"]),
    ).to(device)

    criterion = nn.L1Loss() if p["loss"] == "l1" else (lambda x, y: torch.mean(torch.sqrt((x - y) ** 2 + 1e-6)))
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(p["lr"]), weight_decay=float(p["weight_decay"]))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6) if use_scheduler else None
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    tr_loader, vl_loader = make_loaders(use_aug=use_aug)

    best_psnr = -1.0
    for _ in tqdm(range(epochs), desc=f"ablation:{name}"):
        _ = train_one_epoch(model, tr_loader, optimizer, criterion, device, scaler=scaler, use_amp=True)
        _, val_psnr = evaluate(model, vl_loader, criterion, device, use_amp=True)
        best_psnr = max(best_psnr, val_psnr)
        if scheduler is not None:
            scheduler.step(val_psnr)

    del tr_loader, vl_loader, model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return best_psnr


ablation_configs = [
    {"name": "baseline_no_aug_no_sched", "use_aug": False, "use_scheduler": False},
    {"name": "aug_only", "use_aug": True, "use_scheduler": False},
    {"name": "aug_plus_scheduler", "use_aug": True, "use_scheduler": True},
]

ablation_rows = []
for ab in ablation_configs:
    score = run_ablation(**ab, epochs=8)
    ablation_rows.append({**ab, "best_psnr": score})

ablation_df = pd.DataFrame(ablation_rows).sort_values("best_psnr", ascending=False).reset_index(drop=True)
display(ablation_df)

ablation_path = os.path.join(cfg.save_dir, "ablation_results.csv")
ablation_df.to_csv(ablation_path, index=False)
print("Saved ablation results to", ablation_path)

In [ ]:
# --------------------------
# Final training: 200 epochs, patience=20, dynamic LR
# --------------------------
best_p = study.best_trial.params
final_model = FastEDSR(
    scale=cfg.scale,
    n_resblocks=int(best_p["n_resblocks"]),
    n_feats=int(best_p["n_feats"]),
    res_scale=float(best_p["res_scale"]),
).to(device)

criterion = nn.L1Loss() if best_p["loss"] == "l1" else (lambda x, y: torch.mean(torch.sqrt((x - y) ** 2 + 1e-6)))
optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=float(best_p["lr"]),
    weight_decay=float(best_p["weight_decay"]),
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=6, min_lr=1e-6
  # dynamic learning rate based on validation PSNR
)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
early_stop = EarlyStopping(patience=cfg.patience, mode="max")

history = []
best_psnr = -1.0
best_path = os.path.join(cfg.save_dir, "best_sr_model.pth")

for epoch in tqdm(range(1, cfg.final_epochs + 1), desc="final-train"):
    train_loss = train_one_epoch(final_model, train_loader, optimizer, criterion, device, scaler=scaler, use_amp=True)
    val_loss, val_psnr = evaluate(final_model, val_loader, criterion, device, use_amp=True)
    scheduler.step(val_psnr)

    current_lr = optimizer.param_groups[0]["lr"]
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_psnr": val_psnr,
        "lr": current_lr,
    })

    if val_psnr > best_psnr:
        best_psnr = val_psnr
        save_model(
            best_path,
            final_model,
            meta={
                "best_psnr": best_psnr,
                "best_params": best_p,
                "cfg": cfg.__dict__,
            },
        )

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_psnr={val_psnr:.3f} | lr={current_lr:.2e}")

    if early_stop.step(val_psnr):
        print(f"Early stopping at epoch {epoch} (patience={cfg.patience}).")
        break

hist_df = pd.DataFrame(history)
hist_path = os.path.join(cfg.save_dir, "training_history.csv")
hist_df.to_csv(hist_path, index=False)
print("Best PSNR:", best_psnr)
print("Best model:", best_path)
print("History CSV:", hist_path)

In [ ]:
# --------------------------
# Load best model, run inference sample, save final artifacts
# --------------------------
loaded = FastEDSR(
    scale=cfg.scale,
    n_resblocks=int(best_p["n_resblocks"]),
    n_feats=int(best_p["n_feats"]),
    res_scale=float(best_p["res_scale"]),
).to(device)

ckpt = torch.load(os.path.join(cfg.save_dir, "best_sr_model.pth"), map_location=device)
loaded.load_state_dict(ckpt["model_state_dict"])
loaded.eval()

# Use a validation sample for visual sanity check
val_lr, val_hr = next(iter(val_loader))
val_lr = val_lr.to(device)

with torch.inference_mode():
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
        sr = loaded(val_lr[:1]).float().clamp(0, 1)

lr_np = val_lr[0].detach().cpu().permute(1, 2, 0).numpy()
hr_np = val_hr[0].detach().cpu().permute(1, 2, 0).numpy()
sr_np = sr[0].detach().cpu().permute(1, 2, 0).numpy()

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(lr_np); ax[0].set_title("LR")
ax[1].imshow(sr_np); ax[1].set_title("SR (Prediction)")
ax[2].imshow(hr_np); ax[2].set_title("HR (Target)")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

# Save artifacts
final_path = os.path.join(cfg.save_dir, "final_sr_model_last.pth")
torch.save({"model_state_dict": final_model.state_dict(), "meta": {"cfg": cfg.__dict__}}, final_path)
print("Saved final model to", final_path)

In [ ]:
# --------------------------
# Quick summary for resume-ready reporting
# --------------------------
print("Suggested resume lines:")
print("1) Built a super-resolution pipeline (LR->HR) on Kaggle with FastEDSR and mixed precision training.")
print("2) Added Bayesian hyperparameter optimization (Optuna) and ablation studies for data augmentation/scheduler impact.")
print("3) Trained with dynamic LR scheduling + early stopping (patience=20), saved reproducible best/final checkpoints and training logs.")

print("\nAll outputs stored in:", cfg.save_dir)